# Onion service correlation: client Guard vs service Guard 

**Question.** If an observer sees only the Guard of the client and the Guard of the onion service, can the two sides be linked by timing?

**Method.**
1. Group the packets of each Guard into *bursts* (same direction, gap <= 0.1 s), keep bursts >= 1000 bytes.
2. Pair client bursts with service bursts that start within a lag window. Pairing is **strictly one-to-one**.
3. Compare the real result with two controls that cannot contain a true correspondence:
   * **time shift**: service trace of the same run moved by 10-50 s;
   * **cross-run**: service trace of a *different* run j, aligned to the start of run i. It keeps a realistic burst structure.
4. Check which requests of the client log are covered by at least one pair, for real data and controls.
5. Sensitivity to the lag window and to packet direction.
6. Diagnose the extra client bursts in run_02 / run_03.

All numbers are timing statistics, not precision/recall and not proof of de-anonymisation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Configuration (change here when the network is recreated) -----------
CLIENT_GUARD_IP  = "10.5.184.229"   # relayngpufjeq
SERVICE_GUARD_IP = "10.5.237.145"   # relaysgufrlnh
RUNS = ["run_01", "run_02", "run_03", "run_04", "run_05"]

BURST_GAP       = 0.1     # s
MIN_BURST_BYTES = 1000
LAG_MIN, LAG_MAX = -0.1, 0.5      # main lag window (s): service start - client start
SHIFTS = [10, 20, 30, 40, 50]     # s, time-shift control

RESULTS = next(p for p in [Path("results"), Path("experiments/onion-correlation/results")] if p.is_dir())
print("Results:", RESULTS.resolve())

## 1. Load packets, bursts and request logs

In [ ]:
def load_packets(csv_path, guard_ip):
    df = pd.read_csv(csv_path)
    df.columns = ["time", "size", "src", "dst"]
    df = df.dropna(subset=["src", "dst"]).sort_values("time").reset_index(drop=True)
    df["dir"]  = np.where(df["src"] == guard_ip, 1, -1)            # +1 leaves the Guard, -1 enters it
    df["peer"] = np.where(df["src"] == guard_ip, df["dst"], df["src"])
    return df

def make_bursts(pk):
    new = (pk["time"].diff() > BURST_GAP) | (pk["dir"] != pk["dir"].shift())
    b = pk.assign(burst=new.cumsum()).groupby("burst").agg(
        start=("time", "min"), bytes=("size", "sum"), dir=("dir", "first"), packets=("size", "count"))
    return b[b["bytes"] >= MIN_BURST_BYTES].reset_index(drop=True)

def load_requests(run):
    req = pd.read_csv(RESULTS / run / "client_requests_log.csv")
    return req[req["success"]].reset_index(drop=True)

packets, bursts, requests = {}, {}, {}
for run in RUNS:
    folder = RESULTS / run / "analysis" / "window_5s"
    packets[run] = {"client":  load_packets(folder / "client_metadata.csv",  CLIENT_GUARD_IP),
                    "service": load_packets(folder / "service_metadata.csv", SERVICE_GUARD_IP)}
    bursts[run]  = {side: make_bursts(packets[run][side]) for side in ("client", "service")}
    requests[run] = load_requests(run)
    print(run, "client bursts:", len(bursts[run]["client"]), "| service bursts:", len(bursts[run]["service"]),
          "| requests:", len(requests[run]))

# time of the first request of each run, used to align runs with each other
T0 = {run: requests[run]["timestamp_start"].min() for run in RUNS}

## 2. One-to-one pairing

Client bursts are scanned in time order. Each takes the **closest still-unused** service burst inside the lag window
(optionally with a direction rule). Unlike the first version, a service burst can never be claimed twice.

`direction`: `any` = ignore, `same` = both bursts must have the same direction, `opposite` = opposite directions.

In [ ]:
def pair_bursts(client, service, lag_min=LAG_MIN, lag_max=LAG_MAX, direction="any", shift=0.0):
    c_start, c_dir = client["start"].to_numpy(), client["dir"].to_numpy()
    order = np.argsort(service["start"].to_numpy() + shift)
    s_start = service["start"].to_numpy()[order] + shift
    s_dir = service["dir"].to_numpy()[order]
    used = np.zeros(len(s_start), dtype=bool)
    out = []
    for i in np.argsort(c_start):
        lo = np.searchsorted(s_start, c_start[i] + lag_min, side="left")
        hi = np.searchsorted(s_start, c_start[i] + lag_max, side="right")
        idx = np.arange(lo, hi)
        idx = idx[~used[idx]]
        if direction == "same":
            idx = idx[s_dir[idx] == c_dir[i]]
        elif direction == "opposite":
            idx = idx[s_dir[idx] != c_dir[i]]
        if idx.size == 0:
            continue
        j = idx[np.argmin(np.abs(s_start[idx] - c_start[i]))]
        used[j] = True
        out.append((c_start[i], s_start[j], s_start[j] - c_start[i]))
    return pd.DataFrame(out, columns=["client_start", "service_start", "lag"])

def covered(pairs, req, lag_min=LAG_MIN, lag_max=LAG_MAX):
    """Requests with at least one pair whose client burst starts inside [start+lag_min, end+lag_max]."""
    starts = np.sort(pairs["client_start"].to_numpy())
    lo = np.searchsorted(starts, req["timestamp_start"].to_numpy() + lag_min, side="left")
    hi = np.searchsorted(starts, req["timestamp_end"].to_numpy() + lag_max, side="right")
    return int((hi > lo).sum())

## 3. Real data vs controls

For each run: real pairs, the time-shift control (mean over 5 shifts) and the cross-run control (service of each other run j != i, aligned by the first request, mean over 4 runs). Coverage is computed in the same way for all three.

In [ ]:
def evaluate(lag_min=LAG_MIN, lag_max=LAG_MAX, direction="any"):
    rows = []
    for ri in RUNS:
        c, s, req = bursts[ri]["client"], bursts[ri]["service"], requests[ri]
        real = pair_bursts(c, s, lag_min, lag_max, direction)
        shifted = [pair_bursts(c, s, lag_min, lag_max, direction, shift=sh) for sh in SHIFTS]
        cross = [pair_bursts(c, bursts[rj]["service"], lag_min, lag_max, direction, shift=T0[ri] - T0[rj])
                 for rj in RUNS if rj != ri]
        rows.append({
            "run": ri, "client_bursts": len(c), "service_bursts": len(s), "n_req": len(req),
            "pairs": len(real),
            "pairs_shift": np.mean([len(p) for p in shifted]),
            "pairs_cross": np.mean([len(p) for p in cross]),
            "cov_real":  covered(real, req, lag_min, lag_max),
            "cov_shift": np.mean([covered(p, req, lag_min, lag_max) for p in shifted]),
            "cov_cross": np.mean([covered(p, req, lag_min, lag_max) for p in cross]),
            "median_lag_s": real["lag"].median(),
        })
        rows[-1]["_lags"] = real["lag"].to_numpy()
    return pd.DataFrame(rows)

def present(res):
    t = pd.DataFrame({
        "run": res["run"],
        "client_bursts": res["client_bursts"],
        "service_bursts": res["service_bursts"],
        "pairs": res["pairs"],
        "paired_%": (100 * res["pairs"] / res["client_bursts"]).round(1),
        "shift_%":  (100 * res["pairs_shift"] / res["client_bursts"]).round(1),
        "cross_run_%": (100 * res["pairs_cross"] / res["client_bursts"]).round(1),
        "median_lag_s": res["median_lag_s"].round(4),
        "req_covered": res["cov_real"].astype(str) + "/" + res["n_req"].astype(str),
        "req_covered_shift": res["cov_shift"].round(1),
        "req_covered_cross": res["cov_cross"].round(1),
    })
    tot = res[["client_bursts", "service_bursts", "pairs", "pairs_shift", "pairs_cross",
               "cov_real", "cov_shift", "cov_cross", "n_req"]].sum()
    t.loc[len(t)] = ["ALL", tot.client_bursts, tot.service_bursts, tot.pairs,
                     round(100 * tot.pairs / tot.client_bursts, 1),
                     round(100 * tot.pairs_shift / tot.client_bursts, 1),
                     round(100 * tot.pairs_cross / tot.client_bursts, 1),
                     round(float(np.median(np.concatenate(res["_lags"].to_list()))), 4),
                     f"{int(tot.cov_real)}/{int(tot.n_req)}", round(tot.cov_shift, 1), round(tot.cov_cross, 1)]
    return t

main = evaluate()
summary = present(main)
summary.to_csv(RESULTS / "onion_summary.csv", index=False)
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
lags = np.concatenate(main["_lags"].to_list())
ax[0].hist(lags, bins=40, color="steelblue", edgecolor="white")
ax[0].axvline(0, color="darkred", ls="--")
ax[0].set(title="Lag of paired bursts", xlabel="service start - client start (s)", ylabel="pairs")

x, per = np.arange(len(main)), summary.iloc[:-1]
ax[1].bar(x - 0.27, per["paired_%"], 0.27, label="real", color="darkorange")
ax[1].bar(x,        per["shift_%"],  0.27, label="time shift", color="lightgray")
ax[1].bar(x + 0.27, per["cross_run_%"], 0.27, label="cross-run", color="gray")
ax[1].set_xticks(x, per["run"]); ax[1].set(title="Client bursts that find a partner", ylabel="%"); ax[1].legend()

n = main["n_req"]
ax[2].bar(x - 0.27, main["cov_real"],  0.27, label="real", color="darkorange")
ax[2].bar(x,        main["cov_shift"], 0.27, label="time shift", color="lightgray")
ax[2].bar(x + 0.27, main["cov_cross"], 0.27, label="cross-run", color="gray")
ax[2].set_xticks(x, main["run"]); ax[2].set(title="Requests covered (out of 100)", ylim=(0, 105)); ax[2].legend()
plt.tight_layout(); plt.show()

## 4. Sensitivity: lag window and packet direction

If the result barely changes with a much tighter window, the wide +0.5 s margin is not needed. The direction rule tells whether using packet direction adds information (`same` / `opposite` is relative to each Guard's own IP). Values are pooled over the five runs.

In [ ]:
rows = []
for (lmin, lmax) in [(-0.1, 0.5), (-0.05, 0.05), (-0.02, 0.02)]:
    for direction in ["any", "same", "opposite"]:
        r = evaluate(lmin, lmax, direction)
        t = r[["client_bursts", "pairs", "pairs_shift", "pairs_cross", "cov_real", "cov_shift", "cov_cross", "n_req"]].sum()
        rows.append({
            "lag_window_s": f"[{lmin}, {lmax}]", "direction": direction,
            "paired_%": round(100 * t.pairs / t.client_bursts, 1),
            "shift_%": round(100 * t.pairs_shift / t.client_bursts, 1),
            "cross_run_%": round(100 * t.pairs_cross / t.client_bursts, 1),
            "req_covered_%": round(100 * t.cov_real / t.n_req, 1),
            "req_covered_cross_%": round(100 * t.cov_cross / t.n_req, 1),
        })
sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(RESULTS / "onion_sensitivity.csv", index=False)
sensitivity

## 5. Why does the client Guard have extra bursts in some runs?

Compare bursts that start **inside** a request window (request interval extended by the lag margins) with bursts **between** requests, plus the number of distinct peers seen at the Guard. Bursts between requests are background traffic: Tor control cells, circuit (re)builds, keep-alives, or other circuits through the same relay.

In [ ]:
def in_request_window(starts, req):
    s = np.sort(starts)
    lo = np.searchsorted(s, req["timestamp_start"].to_numpy() + LAG_MIN, side="left")
    hi = np.searchsorted(s, req["timestamp_end"].to_numpy() + LAG_MAX, side="right")
    mask = np.zeros(len(s), dtype=bool)
    for a, b in zip(lo, hi):
        mask[a:b] = True
    return pd.Series(mask, index=np.argsort(np.argsort(starts)))   # back to original order

diag = []
for run in RUNS:
    for side in ("client", "service"):
        b = bursts[run][side]
        inside = in_request_window(b["start"].to_numpy(), requests[run]).to_numpy()
        pk = packets[run][side]
        diag.append({
            "run": run, "side": side, "bursts": len(b),
            "in_request_window": int(inside.sum()),
            "between_requests": int((~inside).sum()),
            "bursts_per_request": round(inside.sum() / len(requests[run]), 2),
            "median_bytes_inside": int(b.loc[inside, "bytes"].median()) if inside.any() else None,
            "median_bytes_between": int(b.loc[~inside, "bytes"].median()) if (~inside).any() else None,
            "distinct_peer_IPs": pk["peer"].nunique(),
            "packets": len(pk),
        })
diagnostics = pd.DataFrame(diag)
diagnostics.to_csv(RESULTS / "onion_diagnostics.csv", index=False)
diagnostics

In [ ]:
# Bursts per 5-second bin over the run: extra background shows up as a higher or irregular client line
fig, axes = plt.subplots(len(RUNS), 1, figsize=(12, 2.2 * len(RUNS)), sharex=False)
for ax, run in zip(axes, RUNS):
    t0 = min(bursts[run]["client"]["start"].min(), bursts[run]["service"]["start"].min())
    for side, color in (("client", "darkorange"), ("service", "steelblue")):
        s = bursts[run][side]["start"] - t0
        ax.hist(s, bins=np.arange(0, s.max() + 5, 5), histtype="step", lw=1.6, color=color, label=f"{side} Guard")
    ax.set(ylabel=run); ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("time since first burst (s)"); axes[0].set_title("Bursts per 5 s")
plt.tight_layout(); plt.show()

Reading guide for the diagnostics (fill the conclusion in the report once you have seen the numbers):

* **`between_requests` much higher at the client Guard in run_02 / run_03** -> the extra bursts are background traffic, not requests. The other metrics are then unaffected, but `paired_%` is lower simply because the denominator contains unpairable bursts.
* **`in_request_window` also higher / more `bursts_per_request`** -> each request produced more bursts at the client side (e.g. different fragmentation of the same transfer, or a circuit rebuild during the run).
* **More `distinct_peer_IPs`** -> the Guard carried traffic of additional circuits or peers during that capture.
* Optional follow-up: look at the `docker logs` of the client container for the run time (new circuits, `Guard`/`circuit` messages) to confirm the cause.

## 6. Open checks

**6.1 Who talks to each Guard?** Packets per peer IP and run. If the client Guard also carries the service's half of the circuit (e.g. as rendezvous point or middle relay), the service container or the middle relays will show up as heavy peers of the client Guard, which would explain the extra bursts in run_02 / run_03. This is a check, not a conclusion.

In [ ]:
def peer_table(side):
    t = pd.concat({run: packets[run][side].groupby("peer").size() for run in RUNS}, axis=1).fillna(0).astype(int)
    return t.sort_values(RUNS[0], ascending=False)

print("Packets per peer at the CLIENT Guard"); display(peer_table("client"))
print("Packets per peer at the SERVICE Guard"); display(peer_table("service"))

**6.2 Requests without a linked pair.** For each uncovered request: bursts of at least 1000 bytes and all bursts (any size) in its window, at both Guards. If one side has no big burst in the window, the request is missed because of the size filter; if both have bursts but no pair, the timing rule failed.

In [ ]:
rows = []
for run in RUNS:
    req = requests[run]
    real = pair_bursts(bursts[run]["client"], bursts[run]["service"])
    starts = np.sort(real["client_start"].to_numpy())
    all_b = {}
    for s in ("client", "service"):
        pk = packets[run][s]
        new = (pk["time"].diff() > BURST_GAP) | (pk["dir"] != pk["dir"].shift())
        all_b[s] = pk.assign(burst=new.cumsum()).groupby("burst").agg(start=("time", "min"), bytes=("size", "sum"))
    for _, r in req.iterrows():
        lo, hi = r["timestamp_start"] + LAG_MIN, r["timestamp_end"] + LAG_MAX
        if ((starts >= lo) & (starts <= hi)).any():
            continue
        def count(df, min_bytes):
            return int(((df["start"] >= lo) & (df["start"] <= hi) & (df["bytes"] >= min_bytes)).sum())
        rows.append({"run": run, "request_id": int(r["request_id"]),
                     "client_big": count(all_b["client"], MIN_BURST_BYTES), "service_big": count(all_b["service"], MIN_BURST_BYTES),
                     "client_any": count(all_b["client"], 0), "service_any": count(all_b["service"], 0)})
uncovered = pd.DataFrame(rows)
uncovered.to_csv(RESULTS / "onion_uncovered_requests.csv", index=False)
print("Uncovered requests:", len(uncovered))
print("  no big burst on the client side :", int((uncovered["client_big"] == 0).sum()))
print("  no big burst on the service side:", int((uncovered["service_big"] == 0).sum()))
print("  big bursts on both sides, no pair:", int(((uncovered["client_big"] > 0) & (uncovered["service_big"] > 0)).sum()))
uncovered.head(20)

## 7. Limits of this experiment

* **One host, one clock.** All containers run on the same machine with the same clock and almost no network delay, so timing offsets between the two Guards are near zero (median lag ~ 0 s). On the real Tor network, latency and clock differences are much larger.
* **A single tiny page.** The service returns `<h1>Hello Tor</h1>`: every request looks the same, so the traffic is simpler than real browsing.
* **No other users.** One client and one service in a private network, with no real background traffic. The attacker only has to link one pair, not choose among many.
* **Hand-chosen thresholds.** Burst gap (0.1 s), minimum size (1000 bytes) and lag window are not calibrated; the sensitivity table shows how much they matter.
* **Burst size = frame size** at the Guard (TLS/Tor cells included), not HTTP payload size.
* **Request attribution** relies on timestamps; the server log has one-second resolution and bursts are not labelled individually.
* The percentages are **timing statistics**, not precision, recall or accuracy, and they are not proof of de-anonymisation. They show that a timing correlation exists in this setup.